# Exploration — iterations-to-threshold vs chip size `m`

How many `(ML + φ-IFM)` cycles does the iterative characterisation protocol need before the
**post-ML test TVD** drops below a fixed threshold, as a function of chip size `m`?

**Compute is offloaded.** The sweep itself runs in `scripts/sweep_m.py` (headless, saves a JSON
bundle); this notebook only *loads and plots*. So a 20–40 min sweep runs once and every figure
tweak afterwards is instant. Regenerate the data with:

```
uv run python scripts/sweep_m.py --m-values 4 6 8 10 --epochs 80
```

See `scripts/sweep_m.py --help` for the m-grid / epochs / threshold / param-data-ratio knobs.

**Constant parameter:data ratio.** The sweep scales the dataset with the trainable parameter
count: `n_samples = round(param_data_ratio · n_params)`, where
`n_params = n_PS² (C_2) + n_BS (R) + m (T_out)` and `n_PS = n_BS = m(m-1)`. The paper reports
**~1.02 data points per parameter** as optimal, so that is the default. At *fixed* `n_samples`
the larger chips are badly under-determined (`C_2` alone has `n_PS²` entries) — they never
converge and burn the full cycle budget. Holding the ratio constant puts every `m` on the same
statistical footing. Train/test split is 80/20.

**Why post-ML TVD?** The post-φ-IFM TVD oscillates *upward* each cycle — φ-IFM rewrites `c_0`
without re-fitting `C_2`, so the previously-compensating `C_2` is briefly mismatched. The
post-ML TVD is the model's true end-of-cycle quality and the meaningful convergence signal.

In [ ]:
import json
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt

# Project root, regardless of the cwd the kernel was launched from.
_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent

print(f"project root: {_ROOT}")

## Load the sweep bundle

Reads `outputs/sweep_m.json` produced by `scripts/sweep_m.py`. If it's missing, run the sweep
script first (see the command in the intro). The config block (epochs, threshold,
param-data-ratio, max-cycles, seed) is read straight from the bundle so the plots always
describe the data they were built from.

In [ ]:
SWEEP_PATH = _ROOT / "outputs" / "sweep_m.json"
if not SWEEP_PATH.exists():
    raise FileNotFoundError(
        f"{SWEEP_PATH} not found.\n"
        f"Run the sweep first:\n"
        f"    uv run python scripts/sweep_m.py\n"
        f"(see scripts/sweep_m.py --help for the m-grid / epochs / threshold knobs)."
    )

bundle = json.loads(SWEEP_PATH.read_text(encoding="utf-8"))
config = bundle["config"]

# Per-m results, keyed by int m and sorted ascending.
results = {
    int(k): v
    for k, v in sorted(bundle["results"].items(), key=lambda kv: int(kv[0]))
}

THRESHOLD = config["threshold"]
EPOCHS = config["epochs"]
MAX_CYCLES = config["max_cycles"]
PARAM_DATA_RATIO = config["param_data_ratio"]

print(f"loaded {SWEEP_PATH.name}")
print(f"  m_values         = {list(results)}")
print(f"  epochs/ML-stage  = {EPOCHS}  (fixed across m)")
print(f"  param_data_ratio = {PARAM_DATA_RATIO}  (n_samples = ratio x n_params)")
print(f"  threshold        = {THRESHOLD * 100:g}%")
print(f"  max_cycles       = {MAX_CYCLES}")
print(f"  sweep wall-clock = {bundle['total_elapsed']:.1f} s")

In [ ]:
# ---- Summary table -------------------------------------------------------
print(f"{'m':>4}  {'n_PS':>5}  {'n_params':>9}  {'n_samples':>9}  "
      f"{'cycles->thr':>12}  {'final TVD':>11}  {'wall-clock':>11}")
print("-" * 78)
for _m, _r in results.items():
    _c = _r["cycles_to_threshold"]
    _c_str = str(_c) if _c is not None else "DNF"
    _final_tvd = _r["history"][-1]["post_ml_tvd"] * 100
    print(f"{_m:>4}  {_r['n_PS']:>5}  {_r['n_params']:>9}  {_r['n_samples']:>9}  "
          f"{_c_str:>12}  {_final_tvd:>10.4f}%  {_r['elapsed']:>9.1f} s")

## Plot 1 — iterations to threshold vs `m`

The headline comparison: with the parameter:data ratio held constant, how many `(ML + φ-IFM)`
cycles does each chip size need to drive the post-ML TVD below the threshold? Chips that never
reach it within `MAX_CYCLES` are drawn as hatched bars at the budget ceiling (`DNF`).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

ms = list(results.keys())
cycles = [results[m]["cycles_to_threshold"] for m in ms]

converged = [(m, c) for m, c in zip(ms, cycles) if c is not None]
dnf = [m for m, c in zip(ms, cycles) if c is None]

if converged:
    cm, cc = zip(*converged)
    ax.bar(cm, cc, width=1.2, color="C0", alpha=0.85,
           label="cycles to reach threshold")
    for m, c in converged:
        ax.text(m, c + 0.12, str(c), ha="center", va="bottom", fontsize=10)
for i, m in enumerate(dnf):
    ax.bar(m, MAX_CYCLES, width=1.2, color="lightgray", hatch="///",
           edgecolor="gray",
           label="did not reach threshold (DNF)" if i == 0 else None)
    ax.text(m, MAX_CYCLES + 0.12, "DNF", ha="center", va="bottom",
            fontsize=10, color="gray")

ax.set_xlabel("chip size  m")
ax.set_ylabel(f"(ML + \u03c6-IFM) cycles to post-ML TVD < {THRESHOLD * 100:g}%")
ax.set_title(f"Iterations to threshold vs chip size\n"
             f"(epochs/ML-stage = {EPOCHS} fixed;  "
             f"n_samples = {PARAM_DATA_RATIO} \u00d7 n_params)")
ax.set_xticks(ms)
ax.set_ylim(0, MAX_CYCLES + 1)
ax.grid(True, axis="y", alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Plot 2 — TVD convergence trajectories

Context for Plot 1: the full post-ML TVD curve per `m`. Cycle 0 is the V-IFM-seed starting point
(no training yet). The dotted line is the threshold; where each curve first crosses it is the
bar height in Plot 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 5.0))

for i, (m, r) in enumerate(results.items()):
    hist = r["history"]
    cyc = [0] + [h["cycle"] for h in hist]
    tvd = [r["initial_tvd"] * 100] + [h["post_ml_tvd"] * 100 for h in hist]
    ax.semilogy(cyc, tvd, "o-", color=f"C{i}", label=f"m = {m}")

ax.axhline(THRESHOLD * 100, color="gray", linestyle=":",
           label=f"threshold = {THRESHOLD * 100:g}%")
ax.set_xlabel("cycle")
ax.set_ylabel("post-ML test TVD  [%]")
ax.set_title(f"Post-ML TVD convergence per chip size  (epochs/stage = {EPOCHS})")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Notes & next steps

- **What's controlled.** `epochs` per ML stage and the parameter:data ratio are fixed across
  `m`. The dataset size therefore scales as `n_params ≈ n_PS²` (so `~m⁴`), holding every chip at
  the same ~1.02 data-points-per-parameter the paper reports as optimal. This removes the
  data-starvation confound that made the old fixed-`n_samples` sweep's large-`m` chips DNF.
- **Runtime.** Because `n_samples ~ m⁴`, the larger `m` carry most of the wall-clock — but across
  `m ≤ 12` it stays in the low tens of thousands of samples (m=12 ≈ 18k). The `n_params` and
  `n_samples` columns in the summary table make the budget explicit.
- **Reaching the paper's 1e-3.** Re-run the script with `--threshold 1e-3 --max-cycles 15`;
  expect smaller `m` to still converge quickly and larger ones to need more cycles.
- **Seed sensitivity.** One `seed` per sweep. Run `sweep_m.py` a few times with different
  `--seed`, save to distinct `--out` paths, and overlay them for a mean ± spread version of
  Plot 1.
- **m = 12 (paper size).** Add `12` to `--m-values` once the smaller runs look right — it is the
  slowest single point but otherwise identical.

---

# Fig 3.B — iterations-to-threshold vs epochs-per-ML-stage

How many `(ML + φ-IFM)` cycles does the protocol need before the **post-ML test TVD** drops below a fixed threshold, as a function of the number of **epochs per ML stage**?  Chip size `m` and dataset size are held fixed.

Compute is offloaded to `scripts/sweep_epochs.py`. Regenerate with:

```
uv run python scripts/sweep_epochs.py --epochs-values 50 100 200 500 --m 6
```

See `scripts/sweep_epochs.py --help` for all knobs (m, threshold, max-cycles, param-data-ratio, seed, out-path).

In [ ]:
SWEEP_EPOCHS_PATH = _ROOT / "outputs" / "sweep_epochs.json"
if not SWEEP_EPOCHS_PATH.exists():
    raise FileNotFoundError(
        f"{SWEEP_EPOCHS_PATH} not found.\n"
        "Run the sweep first:\n"
        "    uv run python scripts/sweep_epochs.py\n"
        "(see scripts/sweep_epochs.py --help for all knobs)."
    )

eb = json.loads(SWEEP_EPOCHS_PATH.read_text(encoding='utf-8'))
ecfg = eb['config']

# Per-epochs results, sorted ascending.
eresults = {
    int(k): v
    for k, v in sorted(eb['results'].items(), key=lambda kv: int(kv[0]))
}

E_THRESHOLD  = ecfg['threshold']
E_MAX_CYCLES = ecfg['max_cycles']
E_M          = ecfg['m']
E_N_PARAMS   = ecfg['n_params']
E_N_SAMPLES  = ecfg['n_samples']

print(f'loaded {SWEEP_EPOCHS_PATH.name}')
print(f"  m              = {E_M}  (n_PS = {ecfg['n_PS']},  "
      f"n_params = {E_N_PARAMS},  n_samples = {E_N_SAMPLES})")
print(f'  epochs_values  = {list(eresults)}')
print(f'  threshold      = {E_THRESHOLD * 100:g}%')
print(f'  max_cycles     = {E_MAX_CYCLES}')
print(f"  sweep time     = {eb['total_elapsed']:.1f} s")

In [ ]:
# ---- Summary table ---------------------------------------------------
print(f"{'epochs':>8}  {'cycles->thr':>12}  {'final TVD':>11}  {'wall-clock':>11}")
print("-" * 50)
for _e, _r in eresults.items():
    _c = _r['cycles_to_threshold']
    _c_str = str(_c) if _c is not None else 'DNF'
    _final_tvd = (_r['history'][-1]['post_ml_tvd'] * 100
                  if _r['history'] else float('nan'))
    print(f"{_e:>8}  {_c_str:>12}  {_final_tvd:>10.4f}%  {_r['elapsed']:>9.1f} s")

## Plot 1 (Fig 3.B) — cycles to threshold vs epochs per ML stage

The headline result: more epochs per stage means fewer cycles needed, up to the point of diminishing returns. Bars that reach the budget ceiling are hatched as DNF (did not finish within `max_cycles`).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

ep_vals = list(eresults.keys())
cycles  = [eresults[e]['cycles_to_threshold'] for e in ep_vals]

converged_ep = [(e, c) for e, c in zip(ep_vals, cycles) if c is not None]
dnf_ep       = [e for e, c in zip(ep_vals, cycles) if c is None]

bar_w = max(ep_vals) * 0.06 if ep_vals else 1

if converged_ep:
    ce, cc = zip(*converged_ep)
    ax.bar(ce, cc, width=bar_w, color='C0', alpha=0.85,
           label='cycles to reach threshold')
    for e, c in converged_ep:
        ax.text(e, c + 0.12, str(c), ha='center', va='bottom', fontsize=10)

for i, e in enumerate(dnf_ep):
    ax.bar(e, E_MAX_CYCLES, width=bar_w, color='lightgray', hatch='///',
           edgecolor='gray',
           label='did not reach threshold (DNF)' if i == 0 else None)
    ax.text(e, E_MAX_CYCLES + 0.12, 'DNF', ha='center', va='bottom',
            fontsize=10, color='gray')

ax.set_xlabel('epochs per ML stage')
ax.set_ylabel(
    f'(ML + \u03c6-IFM) cycles to post-ML TVD < {E_THRESHOLD * 100:g}%'
)
ax.set_title(
    f'Fig 3.B \u2014 iterations to threshold vs epochs per ML stage\n'
    f'(m = {E_M},  n_samples = {E_N_SAMPLES},  '
    f'threshold = {E_THRESHOLD * 100:g}%)'
)
ax.set_xticks(ep_vals)
ax.set_ylim(0, E_MAX_CYCLES + 1)
ax.grid(True, axis='y', alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Plot 2 — TVD convergence trajectories (per epochs value)

Context for Plot 1: the full post-ML TVD curve for each epochs value. Cycle 0 is the V-IFM seed starting point. The dotted line is the threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 5.0))

for i, (e, r) in enumerate(eresults.items()):
    hist = r['history']
    cyc  = [0] + [h['cycle'] for h in hist]
    tvd  = [r['initial_tvd'] * 100] + [h['post_ml_tvd'] * 100 for h in hist]
    ax.semilogy(cyc, tvd, 'o-', color=f'C{i}', label=f'epochs = {e}')

ax.axhline(E_THRESHOLD * 100, color='gray', linestyle=':',
           label=f'threshold = {E_THRESHOLD * 100:g}%')
ax.set_xlabel('cycle')
ax.set_ylabel('post-ML test TVD  [%]')
ax.set_title(
    f'TVD convergence per epochs value  '
    f'(m = {E_M},  n_samples = {E_N_SAMPLES})'
)
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---

# Fig 3.B (full) — cycles-to-threshold vs epochs, one curve per m

Replicates Figure 3.B of Fyrillas et al. (2024): the number of `(ML + φ-IFM)` cycles needed to drive the post-ML test TVD below the threshold, as a function of epochs per ML stage, for each chip size `m`. Each `m` is a separate curve; the dataset size scales with `m` at fixed parameter:data ratio = 1.02.

Compute is offloaded to `scripts/sweep_m_epochs.py`. Run once:

```
uv run python scripts/sweep_m_epochs.py 2>&1 | tee outputs/sweep_m_epochs.log
```

The script saves incrementally — if interrupted, re-run the same command to resume from the last completed (m, epochs) pair.

In [ ]:
ME_PATH = _ROOT / "outputs" / "sweep_m_epochs.json"
if not ME_PATH.exists():
    raise FileNotFoundError(
        f"{ME_PATH} not found. Run the sweep first:\n"
        "    uv run python scripts/sweep_m_epochs.py"
    )

me = json.loads(ME_PATH.read_text(encoding='utf-8'))
mecfg = me['config']

ME_THRESHOLD  = mecfg['threshold']
ME_MAX_CYCLES = mecfg['max_cycles']
ME_M_VALUES   = mecfg['m_values']
ME_EP_VALUES  = mecfg['epochs_values']

# results[m_str][e_str] -> entry dict
meresults = me['results']

print(f'loaded {ME_PATH.name}  ({me["total_elapsed"]:.1f} s total)')
print(f'  m_values     = {ME_M_VALUES}')
print(f'  epochs_values = {ME_EP_VALUES}')
print(f'  threshold    = {ME_THRESHOLD * 100:g}%')
print(f'  max_cycles   = {ME_MAX_CYCLES}')

In [ ]:
# ---- Summary table ---------------------------------------------------
header = f"{'m':>4}  {'epochs':>8}  {'cycles':>8}  "\
         f"{'final TVD':>12}  {'time (s)':>10}  {'status':>8}"
print(header)
print("-" * len(header))
for m in ME_M_VALUES:
    for e in ME_EP_VALUES:
        entry = meresults.get(str(m), {}).get(str(e), {})
        if not entry:
            print(f'{m:>4}  {e:>8}  {"MISSING":>8}')
            continue
        if entry.get('status') == 'error':
            print(f'{m:>4}  {e:>8}  {"ERROR":>8}')
            continue
        cyc = entry['cycles_to_threshold']
        cyc_str = str(cyc) if cyc is not None else 'DNF'
        hist = entry.get('history', [])
        ftv = hist[-1]['post_ml_tvd'] * 100 if hist else float('nan')
        print(f'{m:>4}  {e:>8}  {cyc_str:>8}  '
              f'{ftv:>11.4f}%  {entry["elapsed"]:>10.1f}  ok')

## Plot 1 (Fig 3.B) — cycles to threshold vs epochs, one curve per m

Each line is one chip size. The x-axis is epochs per ML stage; the y-axis is the number of `(ML + φ-IFM)` cycles to drive the post-ML TVD below the threshold. Hatched points indicate DNF (did not reach threshold within `max_cycles`). This is the primary Fig 3.B replication.

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 5.0))

for i, m in enumerate(ME_M_VALUES):
    ep_x, cyc_y, dnf_x = [], [], []
    for e in ME_EP_VALUES:
        entry = meresults.get(str(m), {}).get(str(e), {})
        if not entry or entry.get('status') == 'error':
            continue
        c = entry['cycles_to_threshold']
        if c is not None:
            ep_x.append(e)
            cyc_y.append(c)
        else:
            dnf_x.append(e)
    color = f'C{i}'
    if ep_x:
        ax.plot(ep_x, cyc_y, 'o-', color=color, label=f'm = {m}')
        for x, y in zip(ep_x, cyc_y):
            ax.annotate(str(y), (x, y), textcoords='offset points',
                        xytext=(0, 6), ha='center', fontsize=8, color=color)
    for x in dnf_x:
        ax.plot(x, ME_MAX_CYCLES, 'x', color=color, markersize=10,
                markeredgewidth=2,
                label=f'm = {m} (DNF)' if not ep_x else None)

ax.set_xlabel('epochs per ML stage')
ax.set_ylabel(
    f'(ML + \u03c6-IFM) cycles to post-ML TVD < {ME_THRESHOLD * 100:g}%'
)
ax.set_title(
    f'Fig 3.B \u2014 iterations to threshold vs epochs per ML stage\n'
    f'(threshold = {ME_THRESHOLD * 100:g}%,  \'\n'
    f'param:data ratio = {mecfg["param_data_ratio"]})'
)
ax.set_xticks(ME_EP_VALUES)
ax.set_ylim(0, ME_MAX_CYCLES + 1)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Plot 2 — TVD convergence trajectories (all m, all epochs)

One subplot per `m`. Within each subplot, curves show the post-ML TVD trajectory for each epochs value. Lets you see how quickly each chip size converges and whether more epochs per stage makes a qualitative difference.

In [ ]:
n_m = len(ME_M_VALUES)
fig, axes = plt.subplots(1, n_m, figsize=(4.5 * n_m, 4.5), sharey=True)
if n_m == 1:
    axes = [axes]

for ax, m in zip(axes, ME_M_VALUES):
    for j, e in enumerate(ME_EP_VALUES):
        entry = meresults.get(str(m), {}).get(str(e), {})
        if not entry or entry.get('status') == 'error':
            continue
        hist = entry.get('history', [])
        cyc  = [0] + [h['cycle'] for h in hist]
        tvd  = [entry['initial_tvd'] * 100] + [
            h['post_ml_tvd'] * 100 for h in hist]
        ax.semilogy(cyc, tvd, 'o-', color=f'C{j}', label=f'{e} ep')
    ax.axhline(ME_THRESHOLD * 100, color='gray', linestyle=':',
               label=f'{ME_THRESHOLD*100:g}%')
    ax.set_title(f'm = {m}  (n_PS = {m*(m-1)})')
    ax.set_xlabel('cycle')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel('post-ML test TVD  [%]')
fig.suptitle(
    f'TVD convergence trajectories  (threshold = {ME_THRESHOLD*100:g}%)',
    y=1.02
)
plt.tight_layout()
plt.show()

## Plot 3 — wall-clock time per (m, epochs) pair

Shows the compute cost of each grid cell as a heat-map table. Useful for understanding where the runtime is spent and planning future sweeps.

In [ ]:
import numpy as np

time_matrix = np.full((len(ME_M_VALUES), len(ME_EP_VALUES)), np.nan)
for i, m in enumerate(ME_M_VALUES):
    for j, e in enumerate(ME_EP_VALUES):
        entry = meresults.get(str(m), {}).get(str(e), {})
        if entry and entry.get('status') == 'ok':
            time_matrix[i, j] = entry['elapsed'] / 60  # minutes

fig, ax = plt.subplots(figsize=(6.0, 3.5))
im = ax.imshow(time_matrix, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='wall-clock time  [min]')
ax.set_xticks(range(len(ME_EP_VALUES)))
ax.set_xticklabels(ME_EP_VALUES)
ax.set_yticks(range(len(ME_M_VALUES)))
ax.set_yticklabels([f'm = {m}' for m in ME_M_VALUES])
ax.set_xlabel('epochs per ML stage')
ax.set_title('Wall-clock time per (m, epochs) pair  [min]')
for i in range(len(ME_M_VALUES)):
    for j in range(len(ME_EP_VALUES)):
        v = time_matrix[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                    fontsize=9, color='black' if v < time_matrix[~np.isnan(time_matrix)].max()*0.6 else 'white')
        else:
            ax.text(j, i, 'N/A', ha='center', va='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

## Plot 4  Per-epoch TVD trajectory across the sweep grid

One subplot per `epochs/stage`, all `m` overlaid. Each cycle's ML stage is drawn as a connected polyline of the per-epoch samples logged every `epochs // 5` steps. Cycle `k` occupies the interval `((k-1)*epochs, k*epochs]` on the x axis (the log prints `epoch X/N` where X = epochs completed so far in the stage, so cycle 1's last sample sits at x = epochs, cycle 2's first sample at x = epochs+40, etc.). The φ-IFM jump is drawn explicitly as a thin dashed vertical from `post-ML TVD` to `post-φ-IFM TVD` at the boundary (the **discontinuity** — the ML curve does not connect to the next cycle directly), with an `x` marker at the new post-φ-IFM value. From that `x` marker a solid segment reconnects to the first ML sample of the next cycle, showing how fast the next ML stage descends from the c_0-perturbed starting point.


In [ ]:
# >>> per-epoch TVD trajectory cell <<<
# Per-epoch test-TVD trajectory, sampled from the .log file (every
# epochs//5 steps -- that's what the sweep prints; the JSON only stores
# per-cycle endpoints, see Plot 2 above).
import re
from collections import defaultdict

ME_LOG_PATH = ME_PATH.with_suffix('.log')
if not ME_LOG_PATH.exists():
    raise FileNotFoundError(
        f'{ME_LOG_PATH} not found.  Set ME_PATH (cell above) to the JSON\n'
        f'whose sidecar log you want to plot.'
    )

_HEADER_RE  = re.compile(r'\[\d+/\d+\]\s+m=(\d+)\s+epochs=(\d+)')
_CYCLE_RE   = re.compile(r'---\s+Cycle\s+(\d+)')
_EPOCH_RE   = re.compile(r'epoch\s+(\d+)/\d+\s+.*test_tvd=([\d.eE+\-]+)%')
_POSTML_RE  = re.compile(r'post-ML\s*:\s*TVD\s*=\s*([\d.eE+\-]+)%')
_POSTPHI_RE = re.compile(r'post-phi-IFM\s*:\s*TVD\s*=\s*([\d.eE+\-]+)%')

# traj[(m, e)][cycle] -> dict with 'samples', 'post_ml', 'post_phi_ifm'
traj: dict = defaultdict(dict)
cur_m = cur_e = cur_cyc = None
for line in ME_LOG_PATH.read_text(encoding='utf-8').splitlines():
    h = _HEADER_RE.search(line)
    if h:
        cur_m, cur_e = int(h.group(1)), int(h.group(2))
        cur_cyc = None
        continue
    c = _CYCLE_RE.search(line)
    if c and cur_m is not None:
        cur_cyc = int(c.group(1))
        traj[(cur_m, cur_e)].setdefault(cur_cyc, {
            'samples': [], 'post_ml': None, 'post_phi_ifm': None,
        })
        continue
    if cur_cyc is None:
        continue
    em = _EPOCH_RE.search(line)
    if em:
        traj[(cur_m, cur_e)][cur_cyc]['samples'].append(
            (int(em.group(1)), float(em.group(2)))
        )
        continue
    pm = _POSTML_RE.search(line)
    if pm:
        traj[(cur_m, cur_e)][cur_cyc]['post_ml'] = float(pm.group(1))
        continue
    pp = _POSTPHI_RE.search(line)
    if pp:
        traj[(cur_m, cur_e)][cur_cyc]['post_phi_ifm'] = float(pp.group(1))
        continue

n_e = len(ME_EP_VALUES)
fig, axes = plt.subplots(1, n_e, figsize=(4.5 * n_e, 4.5), sharey=True)
if n_e == 1:
    axes = [axes]

for ax, e in zip(axes, ME_EP_VALUES):
    max_cycle = 0
    for k, m in enumerate(ME_M_VALUES):
        cell = traj.get((m, e), {})
        if not cell:
            continue
        color = f'C{k}'
        labelled = False
        sorted_cycs = sorted(cell)
        for cyc in sorted_cycs:
            data = cell[cyc]
            samples = data['samples']
            if not samples:
                continue
            xs = [(cyc - 1) * e + ep for ep, _ in samples]
            ys = [tvd for _, tvd in samples]
            ax.semilogy(
                xs, ys, '.-', color=color, markersize=4, linewidth=1.2,
                label=(None if labelled else f'm = {m}'),
            )
            labelled = True
            # phi-IFM jump at the cycle boundary, only if a next cycle exists
            # in this run (the last cycle has no following phi-IFM-fed stage).
            post_ml = data['post_ml'] if data['post_ml'] is not None else ys[-1]
            post_phi = data['post_phi_ifm']
            next_cyc_has_data = (
                (cyc + 1) in cell and cell[cyc + 1]['samples']
            )
            if post_phi is not None and next_cyc_has_data:
                x_b = cyc * e
                # Vertical dashed 'spike' from end-of-ML up/down to
                # post-phi-IFM (the discontinuity).
                ax.plot(
                    [x_b, x_b], [post_ml, post_phi],
                    ':', color=color, linewidth=0.9, alpha=0.55, zorder=2,
                )
                # 'x' marker at the post-phi-IFM TVD.
                ax.plot(
                    x_b, post_phi, marker='x', color=color,
                    markersize=8, markeredgewidth=1.5, zorder=3,
                )
                # Connector from the x marker to the first sample of the
                # next cycle: the ML stage of cycle k+1 starts from the
                # phi-IFM-perturbed state and descends from there.
                next_ep, next_tvd = cell[cyc + 1]['samples'][0]
                next_x = cyc * e + next_ep
                ax.plot(
                    [x_b, next_x], [post_phi, next_tvd],
                    '-', color=color, linewidth=1.2, alpha=0.9, zorder=2,
                )
            max_cycle = max(max_cycle, cyc)
    # Cycle-boundary guides (one per epoch length per subplot).
    for c in range(1, max_cycle):
        ax.axvline(
            x=c * e, color='lightgray', linestyle='-',
            linewidth=0.7, zorder=0,
        )
    ax.axhline(
        ME_THRESHOLD * 100, color='gray', linestyle=':',
        linewidth=0.8, label=f'{ME_THRESHOLD*100:g}%',
    )
    ax.set_title(f'epochs/stage = {e}')
    ax.set_xlabel('cumulative epoch')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')

axes[0].set_ylabel('test TVD  [%]')
fig.suptitle(
    f'Per-epoch test TVD trajectory  '
    f'({ME_LOG_PATH.name})  '
    f'\u2014 dashed spike + \u00d7 = post-\u03c6-IFM jump',
    y=1.02,
)
plt.tight_layout()
plt.show()


## Plot 5  Noise sweep -- TVD floor & per-noise trajectory

Two views of the `sweep_noise.json` + `.log` produced by `scripts/sweep_noise.py`:

* **5a (floor vs noise):** the post-ML TVD reached at the final cycle of each sigma run, plotted against sigma on a log-log axis. A `y = sigma` reference line lets you see whether the TVD floor scales linearly with the injected Gaussian noise (the expectation for normalised-intensity output). Maps qualitatively to paper Fig 3(d).

* **5b (trajectory per sigma):** one subplot per noise level, same per-epoch sampling as Plot 4 (every `epochs // 5` epochs). Cycle boundaries drawn as thin vertical guides; the phi-IFM jump is shown as a dashed spike + 'x' marker. Lets you confirm whether each TVD trajectory has actually **settled** at the floor or is still descending when the run exits.


In [ ]:
# >>> noise-sweep analysis cell <<<
# Loads outputs/sweep_noise.json (+ sidecar .log for per-epoch
# trajectory). Robust to a single-sigma JSON: 5a shows one point
# and 5b shows one subplot.
import re
from collections import defaultdict
import numpy as np

SWEEP_NOISE_PATH = _ROOT / 'outputs' / 'sweep_noise.json'
if not SWEEP_NOISE_PATH.exists():
    raise FileNotFoundError(
        f'{SWEEP_NOISE_PATH} not found. Run:\n'
        '    uv run python scripts/sweep_noise.py --num-threads 4'
    )

ns = json.loads(SWEEP_NOISE_PATH.read_text(encoding='utf-8'))
ns_cfg = ns['config']
# results dict: key = noise sigma as string ('0', '1.000e-04', ...).
# Sort numerically by the per-entry 'noise_std' field.
ns_results_sorted = sorted(
    [
        (k, v) for k, v in ns['results'].items()
        if v.get('status') == 'ok'
    ],
    key=lambda kv: kv[1]['noise_std'],
)

print(f"loaded {SWEEP_NOISE_PATH.name}  "
      f"({len(ns_results_sorted)} completed sigma points, "
      f"total wall-clock {ns['total_elapsed']:.1f} s)")
print(f"  m = {ns_cfg['m']},  epochs/stage = {ns_cfg['epochs']},  "
      f"max_cycles = {ns_cfg['max_cycles']},  "
      f"lr_schedule = {ns_cfg['lr_schedule']}")

# ---- Plot 5a: TVD floor vs sigma ---------------------------------------
sigmas = np.array([entry['noise_std'] for _, entry in ns_results_sorted])
final_tvds = np.array([
    (entry['history'][-1]['post_ml_tvd'] * 100) if entry['history'] else np.nan
    for _, entry in ns_results_sorted
])
cycles_used = np.array([
    len(entry['history']) for _, entry in ns_results_sorted
])

fig, ax = plt.subplots(figsize=(7.0, 5.0))
# Replace sigma=0 with a small placeholder for the log axis.
sigmas_plot = np.where(sigmas == 0, sigmas[sigmas > 0].min() / 10
                       if (sigmas > 0).any() else 1e-6,
                       sigmas)
ax.loglog(sigmas_plot, final_tvds, 'o-', color='C0', markersize=8,
          label='post-ML TVD at final cycle')
# y = sigma reference (in percent: y = sigma * 100)
if (sigmas > 0).any():
    s_ref = np.array([sigmas[sigmas > 0].min(), sigmas.max()])
    ax.loglog(s_ref, s_ref * 100, 'k--', linewidth=0.8, alpha=0.6,
              label=r'$y = \sigma$ (reference)')
ax.axhline(ns_cfg['threshold'] * 100, color='gray', linestyle=':',
           linewidth=0.8, label=f"threshold = {ns_cfg['threshold']*100:g}%")
for s, t, c in zip(sigmas_plot, final_tvds, cycles_used):
    ax.annotate(f'{c} cyc', (s, t), textcoords='offset points',
                xytext=(6, -8), fontsize=8, color='C0')
ax.set_xlabel(r'Gaussian noise $\sigma$  (0 plotted at $\sigma_{\min}/10$)')
ax.set_ylabel('final post-ML test TVD  [%]')
ax.set_title(f'TVD floor vs measurement noise  (m={ns_cfg["m"]}, '
             f'{ns_cfg["lr_schedule"]} LR)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

# ---- Plot 5b: per-sigma per-epoch trajectory ---------------------------
SWEEP_NOISE_LOG = SWEEP_NOISE_PATH.with_suffix('.log')
if not SWEEP_NOISE_LOG.exists():
    print(f'  (skipping 5b: {SWEEP_NOISE_LOG} not found)')
else:
    _HEADER_NOISE_RE = re.compile(
        r'\[\d+/\d+\]\s+sigma=([\d.eE+\-]+)'
    )
    _CYCLE_RE = re.compile(r'---\s+Cycle\s+(\d+)')
    _EPOCH_RE = re.compile(
        r'epoch\s+(\d+)/\d+\s+.*test_tvd=([\d.eE+\-]+)%'
    )
    _POSTML_RE = re.compile(
        r'post-ML\s*:\s*TVD\s*=\s*([\d.eE+\-]+)%'
    )
    _POSTPHI_RE = re.compile(
        r'post-phi-IFM\s*:\s*TVD\s*=\s*([\d.eE+\-]+)%'
    )

    # traj[sigma_float][cycle] -> {'samples', 'post_ml', 'post_phi_ifm'}
    traj: dict = defaultdict(dict)
    cur_sigma = cur_cyc = None
    for line in SWEEP_NOISE_LOG.read_text(encoding='utf-8').splitlines():
        h = _HEADER_NOISE_RE.search(line)
        if h:
            cur_sigma = float(h.group(1))
            cur_cyc = None
            continue
        c = _CYCLE_RE.search(line)
        if c and cur_sigma is not None:
            cur_cyc = int(c.group(1))
            traj[cur_sigma].setdefault(cur_cyc, {
                'samples': [], 'post_ml': None, 'post_phi_ifm': None,
            })
            continue
        if cur_cyc is None:
            continue
        em = _EPOCH_RE.search(line)
        if em:
            traj[cur_sigma][cur_cyc]['samples'].append(
                (int(em.group(1)), float(em.group(2)))
            )
            continue
        pm = _POSTML_RE.search(line)
        if pm:
            traj[cur_sigma][cur_cyc]['post_ml'] = float(pm.group(1))
            continue
        pp = _POSTPHI_RE.search(line)
        if pp:
            traj[cur_sigma][cur_cyc]['post_phi_ifm'] = float(pp.group(1))
            continue

    sigma_keys = sorted(traj.keys())
    n_s = len(sigma_keys)
    if n_s == 0:
        print('  (no trajectory data parsed from log)')
    else:
        fig, axes = plt.subplots(
            1, n_s, figsize=(4.5 * n_s, 4.5), sharey=True,
        )
        if n_s == 1:
            axes = [axes]
        e = ns_cfg['epochs']
        for ax, sigma in zip(axes, sigma_keys):
            cell = traj[sigma]
            max_cycle = max(cell.keys())
            color = 'C0'
            for cyc in sorted(cell):
                d = cell[cyc]
                if not d['samples']:
                    continue
                xs = [(cyc - 1) * e + ep for ep, _ in d['samples']]
                ys = [tvd for _, tvd in d['samples']]
                ax.semilogy(xs, ys, '.-', color=color, markersize=4,
                            linewidth=1.2)
                # phi-IFM spike + connector to next cycle
                post_ml = d['post_ml'] if d['post_ml'] is not None else ys[-1]
                post_phi = d['post_phi_ifm']
                nxt = cell.get(cyc + 1)
                if post_phi is not None and nxt and nxt['samples']:
                    x_b = cyc * e
                    ax.plot([x_b, x_b], [post_ml, post_phi], ':',
                            color=color, linewidth=0.9, alpha=0.55, zorder=2)
                    ax.plot(x_b, post_phi, marker='x', color=color,
                            markersize=8, markeredgewidth=1.5, zorder=3)
                    next_ep, next_tvd = nxt['samples'][0]
                    next_x = cyc * e + next_ep
                    ax.plot([x_b, next_x], [post_phi, next_tvd], '-',
                            color=color, linewidth=1.2, alpha=0.9, zorder=2)
            for c in range(1, max_cycle):
                ax.axvline(x=c * e, color='lightgray', linestyle='-',
                           linewidth=0.7, zorder=0)
            # noise-floor reference (sigma in percent)
            if sigma > 0:
                ax.axhline(sigma * 100, color='red', linestyle=':',
                           linewidth=0.8, alpha=0.6,
                           label=fr'$\sigma$={sigma:.0e} ({sigma*100:g}%)')
            ax.axhline(ns_cfg['threshold'] * 100, color='gray',
                       linestyle=':', linewidth=0.8,
                       label=f"thr={ns_cfg['threshold']*100:g}%")
            ax.set_title(fr'$\sigma$ = {sigma:.0e}')
            ax.set_xlabel('cumulative epoch')
            ax.grid(True, which='both', alpha=0.3)
            ax.legend(fontsize=8, loc='upper right')
        axes[0].set_ylabel('test TVD  [%]')
        fig.suptitle(
            f'Per-epoch TVD trajectory by noise level  '
            f'({SWEEP_NOISE_LOG.name})',
            y=1.02,
        )
        plt.tight_layout()
        plt.show()


---

# Single ML-stage run — `run_synthetic`

One ML stage on a Clements mesh of size `m`. No φ-IFM, no outer loop. Useful for inspecting per-epoch convergence of the ML stage in isolation.

Compute is offloaded to `scripts/run_synthetic.py`. Regenerate with:

```
uv run python scripts/run_synthetic.py --m 6 --epochs 200
```

Output: `outputs/run_synthetic_m<M>.json` (loaded below) and `outputs/run_synthetic_m<M>.log`.

In [ ]:
# Auto-pick the most recently modified outputs/run_synthetic_m*.json.
# Override by setting RS_PATH explicitly, e.g. RS_PATH = _ROOT / "outputs" / "run_synthetic_m12.json"
_candidates = sorted(
    (_ROOT / "outputs").glob("run_synthetic_m*.json"),
    key=lambda p: p.stat().st_mtime,
)
if not _candidates:
    raise FileNotFoundError(
        f"No run_synthetic_m*.json under {_ROOT / 'outputs'}.\n"
        f"Run the script first:\n"
        f"    uv run python scripts/run_synthetic.py --m <M>"
    )
RS_PATH = _candidates[-1]

rs = json.loads(RS_PATH.read_text(encoding="utf-8"))
rs_cfg = rs["config"]
rs_hist = rs["history"]
rs_summary = rs["summary"]
RS_M = rs_cfg["m"]

print(f"loaded {RS_PATH.name}  (m={RS_M})")
print(f"  m={rs_cfg['m']}, n_samples={rs_cfg['n_samples']}, "
      f"epochs={rs_cfg['epochs']}, batch_size={rs_cfg['batch_size']}")
print(f"  LRs:  C2={rs_cfg['lr_C2']:.0e}, R={rs_cfg['lr_R']:.0e}, "
      f"T_out={rs_cfg['lr_Tout']:.0e}")
print(f"  best epoch     : {rs_summary['best_epoch']}")
print(f"  best test MSE  : {rs_summary['best_test_mse']:.3e}")
print(f"  final test MSE : {rs_summary['final_test_mse']:.3e}")
print(f"  final test TVD : {rs_summary['final_test_tvd'] * 100:.3f}%")
print(f"  wall-clock     : {rs_summary['elapsed']:.1f} s")

## Plot — train/test MSE and test TVD per epoch

Two panels: log-scale MSE (train vs test) on the left, test TVD in percent on the right. A divergence between train and test curves indicates overfitting; lockstep oscillation late in training is the Adam-effective-LR-drift pattern discussed in CLAUDE.md.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs_x = range(1, rs_cfg["epochs"] + 1)

axes[0].plot(epochs_x, rs_hist["train_mse"], label="train")
axes[0].plot(epochs_x, rs_hist["test_mse"], label="test")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].set_title(f"MSE vs epoch (m={rs_cfg['m']})")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_x, [t * 100 for t in rs_hist["test_tvd"]])
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test TVD (%)")
axes[1].set_title(f"Test TVD vs epoch (m={rs_cfg['m']})")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---

# Full iterative protocol — `run_iterative`

Single end-to-end (ML + φ-IFM) outer loop until TVD < threshold (or budget runs out). Same protocol as the sweeps, but for a single `(m, epochs)` point with full per-cycle history and final truth/recovered parameter tensors saved.

Compute is offloaded to `scripts/run_iterative.py`. Regenerate with:

```
uv run python scripts/run_iterative.py --m 4 --epochs 200
```

Output: `outputs/iterative_m<M>.json` (loaded below) and `outputs/iterative_m<M>.log`.

In [ ]:
# Auto-pick the most recently modified outputs/iterative_m*.json.
# Override by setting RI_PATH explicitly.
_candidates = sorted(
    (_ROOT / "outputs").glob("iterative_m*.json"),
    key=lambda p: p.stat().st_mtime,
)
if not _candidates:
    raise FileNotFoundError(
        f"No iterative_m*.json under {_ROOT / 'outputs'}.\n"
        f"Run the script first:\n"
        f"    uv run python scripts/run_iterative.py --m <M>"
    )
RI_PATH = _candidates[-1]

ri = json.loads(RI_PATH.read_text(encoding="utf-8"))
ri_cfg = ri["config"]
ri_hist = ri["history"]
ri_summary = ri["summary"]
ri_params = ri["parameters"]
RI_M = ri_cfg["m"]

print(f"loaded {RI_PATH.name}  (m={RI_M})")
print(f"  m={ri_cfg['m']}, epochs={ri_cfg['epochs']}, "
      f"n_samples={ri_cfg['n_samples']}, threshold={ri_cfg['threshold']*100:.4f}%")
print(f"  initial TVD          : {ri_summary['initial_tvd']*100:.4f}%")
print(f"  converged            : {ri_summary['converged']}")
print(f"  cycles to threshold  : {ri_summary['cycles_to_threshold']}")
print(f"  exit reason          : {ri_summary['exit_reason']}")
print(f"  wall-clock           : {ri_summary['elapsed']:.1f} s")
print()
print(f"{'cycle':>6}  {'post-ML TVD':>12}  {'post-phi TVD':>12}  "
      f"{'post-ML MSE':>12}  {'method':>8}")
print("-" * 60)
for h in ri_hist:
    print(f"{h['cycle']:>6}  {h['post_ml_tvd']*100:>11.4f}%  "
          f"{h['post_phi_ifm_tvd']*100:>11.4f}%  {h['post_ml_mse']:>12.3e}  "
          f"{h.get('phi_ifm_method', 'fast'):>8}")

## Plot 1 — TVD convergence per cycle (Fig 3.b precursor)

Log-scale TVD vs cycle. Cycle 0 is the V-IFM-seed starting point. Two curves: post-ML (after gradient descent each cycle) and post-φ-IFM (after the c_0 refinement). The dotted line is the convergence threshold.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(8.0, 5.0))

cycles = np.array([h["cycle"] for h in ri_hist])
post_ml = np.array([h["post_ml_tvd"] for h in ri_hist])
post_phi = np.array([h["post_phi_ifm_tvd"] for h in ri_hist])

cycles_init = np.concatenate(([0], cycles))
post_ml_init = np.concatenate(([ri_summary["initial_tvd"]], post_ml))

ax.semilogy(cycles_init, post_ml_init * 100, "o-", color="C0", label="after ML")
ax.semilogy(cycles, post_phi * 100, "s-", color="C2", label=r"after $\phi$-IFM")
ax.axhline(ri_cfg["threshold"] * 100, color="gray", linestyle=":",
           label=f"target = {ri_cfg['threshold']*100:.3f}%")
ax.set_xlabel("cycle")
ax.set_ylabel("test TVD  [%]")
ax.set_title(f"TVD convergence (m={ri_cfg['m']}, epochs/stage={ri_cfg['epochs']})")
ax.grid(True, which="both", alpha=0.3)
ax.set_xticks(cycles_init)
ax.legend()
fig.tight_layout()
plt.show()

## Plot 2 — Recovered vs ground truth per parameter block (Fig 3.c precursor)

Scatter of recovered vs ground-truth values for each learnable block: `C_2` diagonal, `C_2` off-diagonal, `R`, `T_out`, `c_0`. Points on the y = x line are perfectly recovered.

In [ ]:
import numpy as np

def _offdiag(M_list):
    M = np.asarray(M_list)
    n = M.shape[0]
    return M[~np.eye(n, dtype=bool)]

truth = ri_params["truth"]
recov = ri_params["recovered"]
C_2_truth = np.asarray(truth["C_2"])
C_2_recov = np.asarray(recov["C_2"])

blocks = [
    ("C_2 diag",     np.diag(C_2_truth),         np.diag(C_2_recov),         "C0"),
    ("C_2 off-diag", _offdiag(C_2_truth),        _offdiag(C_2_recov),        "C1"),
    ("R",            np.asarray(truth["R"]),     np.asarray(recov["R"]),     "C2"),
    ("T_out",        np.asarray(truth["T_out"]), np.asarray(recov["T_out"]), "C3"),
    ("c_0",          np.asarray(truth["c_0"]),   np.asarray(recov["c_0"]),   "C4"),
]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for label, gt, est, color in blocks:
    ax.scatter(gt, est, s=10, alpha=0.55, color=color, label=label)

all_vals = np.concatenate([gt for _, gt, _, _ in blocks])
lo, hi = all_vals.min(), all_vals.max()
pad = 0.05 * (hi - lo + 1e-12)
ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad],
        "k--", linewidth=0.8, alpha=0.5)
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_xlabel("ground truth")
ax.set_ylabel("recovered")
ax.set_title(f"Per-block parameter recovery (m={ri_cfg['m']})")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
plt.show()

---

# φ-IFM stage demo — `run_phi_ifm`

Single φ-IFM stage applied to a 'plausible' post-ML model (truth `C_2`, `R`, `T_out` plus noisy `c_0`). Recovers per-PS `c_0` offsets by fitting per-shifter phase-sweep fringes.

Compute is offloaded to `scripts/run_phi_ifm.py`. Regenerate with:

```
uv run python scripts/run_phi_ifm.py --m 4
```

Output: `outputs/run_phi_ifm_m<M>.json` (loaded below) and `outputs/run_phi_ifm_m<M>.log`.

In [ ]:
# Auto-pick the most recently modified outputs/run_phi_ifm_m*.json.
# Override by setting RP_PATH explicitly.
_candidates = sorted(
    (_ROOT / "outputs").glob("run_phi_ifm_m*.json"),
    key=lambda p: p.stat().st_mtime,
)
if not _candidates:
    raise FileNotFoundError(
        f"No run_phi_ifm_m*.json under {_ROOT / 'outputs'}.\n"
        f"Run the script first:\n"
        f"    uv run python scripts/run_phi_ifm.py --m <M>"
    )
RP_PATH = _candidates[-1]

rp = json.loads(RP_PATH.read_text(encoding="utf-8"))
rp_cfg = rp["config"]
rp_summary = rp["summary"]
rp_per_ps = rp["per_ps"]
rp_demo = rp["demo_fringe"]
RP_M = rp_cfg["m"]

print(f"loaded {RP_PATH.name}  (m={RP_M})")
print(f"  m={rp_cfg['m']}, n_PS={rp_cfg['n_PS']}, n_BS={rp_cfg['n_BS']}, "
      f"method={rp_cfg['method']}")
print(f"  characterizable PSs        : {rp_summary['n_characterizable']} / {rp_cfg['n_PS']}")
print(f"  RMS true offset (pre-fit)  : {rp_summary['rms_true_offset_mrad']:.2f} mrad")
print(f"  RMS recovery error         : {rp_summary['rms_recovery_error_mrad']:.2f} mrad")
print(f"  demo PS (largest fringe)   : {rp_summary['demo_ps']}  ({rp_summary['demo_ps_location']})")
if rp_summary["uncharacterizable_idx"]:
    print(f"  uncharacterizable indices  : {rp_summary['uncharacterizable_idx']}")
print()
print(f"{'PS':>3}  {'model amp':>9}  {'in':>3}  {'out':>3}  "
      f"{'true (rad)':>11}  {'recov (rad)':>11}  {'err (mrad)':>10}  status")
for r in rp_per_ps:
    err_mrad = 1000 * (r["recovered_offset"] - r["true_offset"])
    print(f"{r['ps']:>3}  {r['model_amplitude']:>9.2e}  "
          f"{r['input_port']:>3}  {r['output_port']:>3}  "
          f"{r['true_offset']:>+11.4f}  {r['recovered_offset']:>+11.4f}  "
          f"{err_mrad:>+10.2f}  {r['status']}")

## Plot 1 — Demo PS fringe (before vs after fit)

Dense model-predicted fringe before the fit (dashed) and after the fit (solid) for the demo PS, overlaid on the synthetic measurement points. The vertical dotted line marks the true phase offset that the fit should recover.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(8.0, 5.0))

phi_dense = np.asarray(rp_demo["phi_dense"])
ax.plot(phi_dense, rp_demo["f_before"], "--", color="C0", alpha=0.75,
        label=r"model before fit ($\delta\phi=0$)")
ax.plot(phi_dense, rp_demo["f_after"], "-", color="C2", linewidth=2.0,
        label=rf"model after fit ($\delta\phi={rp_demo['recovered_offset']:+.3f}$ rad)")
ax.plot(rp_demo["phi_sweep"], rp_demo["intensity_sweep"], "o",
        color="black", markersize=6, label="synthetic measurement")
ax.axvline(rp_demo["true_offset"] % (2 * np.pi), color="gray",
           linestyle=":", alpha=0.6,
           label=rf"true offset = {rp_demo['true_offset']:+.3f} rad")
ax.set_xlabel(rf"intended phase at PS {rp_summary['demo_ps']}   [rad]")
ax.set_ylabel(rf"intensity at output port {rp_demo['output_port']}")
ax.set_title(rf"$\phi$-IFM fringe, PS {rp_summary['demo_ps']}  "
             rf"(in {rp_demo['input_port']} $\to$ out {rp_demo['output_port']})")
ax.set_xlim(0, 2 * np.pi)
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
plt.show()

## Plot 2 — Per-PS recovery (informative PSs only)

Side-by-side bars of the true c_0 offset and the recovered value for each PS that has a non-flat fringe. PSs marked `flat_fringe` are excluded — they sit after the final BS that mixes their waveguide and their phase cancels in `|U|^2`, so c_0 cannot be recovered.

In [ ]:
import numpy as np

informative = [r for r in rp_per_ps if r["status"] == "OK"]
idx = np.array([r["ps"] for r in informative])
true_off = np.array([r["true_offset"] for r in informative])
recov_off = np.array([r["recovered_offset"] for r in informative])

fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = np.arange(idx.size)
width = 0.4
ax.bar(x - width / 2, true_off, width,
       label="true offset", color="C0", alpha=0.75)
ax.bar(x + width / 2, recov_off, width,
       label="recovered", color="C2", alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in idx])
ax.set_xlabel("phase-shifter index (informative only)")
ax.set_ylabel(r"$c_0$ offset  [rad]")
ax.set_title(rf"$c_0$ offset: truth vs $\phi$-IFM recovery  "
             rf"(RMS err = {rp_summary['rms_recovery_error_mrad']:.2f} mrad)")
ax.axhline(0, color="black", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="y")
ax.legend()
fig.tight_layout()
plt.show()

## Light-routing / IFM-protocol visualisation

Show the V-IFM / phi-IFM protocol that `src.routing.generate_ifm_protocol`
emits for a Clements mesh (Section C of the supplement). Each step is
one phase shifter to sweep, with input port, output port, and the bar /
cross / balanced settings to apply on previously-characterized MZIs so
that light reaches the target without uncontrolled self-interference.

Two renderers:
- **ASCII** (`render_protocol_ascii`): consistent with `ChipMesh.to_ascii`,
  no extra deps.
- **Matplotlib** (`draw_step`, `draw_protocol`, `show_protocol`): mirrors
  Fig. 4 / 6 / 7 / 8 of the supplement -- coloured MZI squares, red
  light path, H/T labels for meta-MZIs.

`%matplotlib inline` was set at the top of the notebook, so the figures
render in this cell.

In [ ]:
# Local imports. The kernel may have been launched from notebooks/, so
# nudge sys.path the same way as the earlier cells.
import sys
from pathlib import Path
_NB_ROOT = Path.cwd()
if _NB_ROOT.name == "notebooks":
    _NB_ROOT = _NB_ROOT.parent
if str(_NB_ROOT) not in sys.path:
    sys.path.insert(0, str(_NB_ROOT))

from src.chip_mesh import ChipMesh
from src.pic_graph import build_from_mesh
from src.routing import (
    generate_ifm_protocol,
    render_protocol_ascii,
    render_step_ascii,
    StepKind,
)
from src.routing_viz import draw_mesh, draw_step, draw_protocol, show_protocol

### Build the protocol

Pick a chip size and generate the protocol. Smaller `m` (4 or 6) gives
something you can scan at a glance; the paper's 12-mode chip emits a
much longer protocol.

In [ ]:
M = 6
mesh = ChipMesh.clements(M)
graph = build_from_mesh(mesh)
protocol = generate_ifm_protocol(mesh)

by_kind = {}
for s in protocol:
    by_kind.setdefault(s.kind.value, 0)
    by_kind[s.kind.value] += 1

unreachable = sum(1 for s in protocol if not s.reachable)
print(f"m = {M}: {len(protocol)} steps total")
for k, n in by_kind.items():
    print(f"  {k:13s}: {n}")
print(f"  unreachable  : {unreachable}")

### ASCII view

One frame per step. `R<idx>` is the target PS being characterized,
`G<idx>` is a previously-characterized MZI being used as a router,
`M<idx>` is a meta-MZI head or tail set to balanced, and `===` highlights
the waveguides the light path occupies on that step.

Showing only the first 4 steps to keep the cell output short -- bump the
slice to see more.

In [ ]:
print(render_protocol_ascii(protocol[:4], mesh, graph))

### Single-step matplotlib figure (`draw_step`)

One step rendered as a paper-style diagram. Pick a step index, supply
the set of MZIs characterized *before* this step (so green / blue
colouring is correct), and call `draw_step`.

In [ ]:
import matplotlib.pyplot as plt

# Show step 0 (a direct-path diagonal) and the first meta-MZI step.
step_indices = [0]
for i, s in enumerate(protocol):
    if s.kind == StepKind.META_MZI and s.reachable:
        step_indices.append(i)
        break

# Build the running characterized-MZI set so each frame is coloured
# correctly relative to the steps that precede it.
characterized = set()
for i in range(max(step_indices) + 1):
    s = protocol[i]
    if i in step_indices:
        fig, ax = plt.subplots(figsize=(2 + 0.6 * len(mesh.layers) / 4,
                                        0.7 * mesh.m + 1))
        draw_step(s, graph, mesh,
                  characterized_before=set(characterized), ax=ax)
        plt.show()
    if s.kind != StepKind.META_MZI:
        characterized.add(s.ps_node_id)

### Grid view of the whole protocol (`draw_protocol`)

One subplot per step, in protocol order. The function tracks the
running characterized set internally so each frame's colours are right.

For chips larger than `m ~ 8` the grid gets unwieldy -- consider slicing
`protocol[:N]` to view a prefix.

In [ ]:
fig = draw_protocol(protocol, graph, mesh, cols=4)
# Optional: save to disk for inclusion in writeups / slides.
# fig.savefig(_NB_ROOT / "outputs" / f"protocol_m{M}.png", dpi=150, bbox_inches="tight")
plt.show()

### Just the bare mesh (`draw_mesh`)

Useful when you want a clean PIC drawing without any step-specific
colouring -- e.g. for a methods diagram.

In [ ]:
fig, ax = plt.subplots(figsize=(2 + 0.6 * len(mesh.layers) / 4,
                                0.7 * mesh.m + 1))
draw_mesh(mesh, ax=ax, graph=graph)
plt.show()